# 07 — Deploy do KNN de preço da corrida

Este notebook adapta `RETREINO/05_deploy.py` ao catálogo `desloc`. Fluxo: carregar a Gold ML → treinar o mesmo pipeline da aula → registrar uma versão no Unity Catalog → testar a versão → criar e consultar um endpoint.

**Uso acadêmico:** as features `dia`, `dia_semana`, `mes` e `hora` vêm de `datahora_entregue`; elas só existem depois da corrida. Por isso, este endpoint serve como demonstração com dados históricos e **não deve ser usado para cotar uma corrida antes de ela acontecer**. O notebook 08 detalha a avaliação e o retreino.

## 1. Importações e dados de treino

Equivale ao `%run .../00_struct_table` do professor, mas lê a tabela permanente criada pelos notebooks 01–03.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
import mlflow
import mlflow.sklearn
import inspect

mlflow.autolog(disable=True)
mlflow.set_registry_uri("databricks-uc")

TABELA_ML = "desloc.gold.transacoes_ml"
NOME_MODELO = "desloc.models.preco_corrida_knn"
NOME_ENDPOINT = "desloc-preco-corrida-knn"

df = spark.table(TABELA_ML).toPandas()
if df.empty:
    raise ValueError(f"A tabela {TABELA_ML} está vazia.")
X = df.drop(columns=["vlr_pago"])
Y = df["vlr_pago"]
print(f"Linhas: {len(df)} | Entradas: {list(X.columns)}")

## 2. Mesmo pipeline da aula

`distancia` no script do professor corresponde a `distancia_lalo` na Gold ML deste projeto. `periodo` é descartado antes do `MinMaxScaler`, como na aula.

In [2]:
colunas_minmax = [
    "dia", "dia_semana", "mes", "hora", "distancia_lalo",
    "latitude_origem", "longitude_origem", "latitude_destino", "longitude_destino"
]
colunas_esperadas = ["periodo", *colunas_minmax]
if set(X.columns) != set(colunas_esperadas):
    raise ValueError(f"Colunas divergentes. Esperadas: {colunas_esperadas}; recebidas: {list(X.columns)}")
X = X[colunas_esperadas]

preprocessador = ColumnTransformer(
    transformers=[
        ("drop", "drop", ["periodo"]),
        ("minmax", MinMaxScaler(), colunas_minmax)
    ],
    remainder="passthrough"
)
pipeline = Pipeline(steps=[
    ("preprocessamento", preprocessador),
    ("knn", KNeighborsRegressor(n_neighbors=5))
])

## 3. Experimento, assinatura e versão registrada

Cria **uma execução** do MLflow, guarda uma amostra de entrada e registra a versão no Unity Catalog. A assinatura é inferida das entradas e saídas reais, evitando tipos incorretos no endpoint. Cada execução desta célula cria uma nova versão do modelo.

In [ ]:
usuario = spark.sql("SELECT current_user()").first()[0]
NOME_EXPERIMENTO = f"/Users/{usuario}/desloc-preco-corrida"
mlflow.set_experiment(NOME_EXPERIMENTO)

pipeline.fit(X, Y)
exemplo_entrada = X.head(5)
signature = infer_signature(exemplo_entrada, pipeline.predict(exemplo_entrada))

with mlflow.start_run(run_name="knn-deploy-aula") as run:
    mlflow.log_table(exemplo_entrada, "train_sample.json")
    argumentos = {
        "sk_model": pipeline,
        "signature": signature,
        "input_example": exemplo_entrada.head(1),
        "registered_model_name": NOME_MODELO,
    }
    parametro_caminho = "name" if "name" in inspect.signature(mlflow.sklearn.log_model).parameters else "artifact_path"
    argumentos[parametro_caminho] = "model"
    info_modelo = mlflow.sklearn.log_model(**argumentos)
    RUN_ID = run.info.run_id

print(f"Experimento: {NOME_EXPERIMENTO}")
print(f"Run ID: {RUN_ID}")
print(f"Modelo registrado: {NOME_MODELO}")

## 4. Recuperar e testar a versão criada

A busca usa o `run_id` desta execução para não escolher acidentalmente uma versão anterior. O teste local usa as mesmas cinco entradas históricas registradas no MLflow.

In [4]:
cliente_mlflow = MlflowClient()
versoes = [
    v for v in cliente_mlflow.search_model_versions(f"name = '{NOME_MODELO}'")
    if v.run_id == RUN_ID
]
if len(versoes) != 1:
    raise RuntimeError(f"Esperava 1 versão para o run {RUN_ID}; encontrei {len(versoes)}.")
VERSAO_MODELO = str(versoes[0].version)
URI_MODELO = f"models:/{NOME_MODELO}/{VERSAO_MODELO}"
modelo = mlflow.pyfunc.load_model(URI_MODELO)
previsoes = modelo.predict(exemplo_entrada)
print(f"Versão: {VERSAO_MODELO} | URI: {URI_MODELO}")
display(exemplo_entrada.assign(previsao_vlr_pago=previsoes))

Versão: 1 | URI: models:/desloc.models.preco_corrida_knn/1


periodo,dia,dia_semana,mes,hora,distancia_lalo,latitude_origem,longitude_origem,latitude_destino,longitude_destino,previsao_vlr_pago
1,3,2,11,7,4.2078524,-23.513742609587695,-46.62327810242303,-23.483853341186528,-46.648585835169115,25.35
1,23,5,10,9,4.3162794,-23.51207921873963,-46.62739562527016,-23.474032512968865,-46.63578672625711,23.318
1,22,4,10,11,4.215667,-23.574913898123835,-46.60181176389887,-23.611927332177384,-46.592856334210495,23.348000000000003
3,18,7,10,18,3.4003665,-23.581033986174024,-46.63696302378459,-23.598154937550955,-46.66461159876348,21.740000000000002
1,13,2,10,11,3.5976384,-23.581544813632622,-46.643830193209304,-23.56791182574625,-46.611816490410796,21.026


## 5. Criar o endpoint de demonstração

A célula abaixo **cria um recurso de Model Serving** e consome a cota do workspace. Defina `CRIAR_ENDPOINT = True` somente quando quiser executar essa etapa. Se o endpoint já existir, atualize a versão dele pela interface do Databricks; este notebook não a altera automaticamente. Na Free Edition, a disponibilidade depende da cota restante.

In [ ]:
CRIAR_ENDPOINT = False

if CRIAR_ENDPOINT:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

    workspace = WorkspaceClient()  # identidade do notebook; nenhum token no código
    endpoint = workspace.serving_endpoints.create(
        name=NOME_ENDPOINT,
        config=EndpointCoreConfigInput(
            name=NOME_ENDPOINT,
            served_entities=[
                ServedEntityInput(
                    name="knn-preco-corrida",
                    entity_name=NOME_MODELO,
                    entity_version=VERSAO_MODELO,
                    workload_size="Small",
                    scale_to_zero_enabled=True,
                )
            ]
        ),
    )
    print(f"Endpoint solicitado: {endpoint.name}")
else:
    print("Criação do endpoint desativada. Revise o modelo e a cota antes de ativar.")

## 6. Consultar o endpoint

Depois que o endpoint estiver **Ready**, teste com dados históricos. O `input_example` registrado permite também usar **Serving → endpoint → Query endpoint → Show Example** na interface, sem token. A célula usa a identidade do notebook pelo SDK.

In [ ]:
CONSULTAR_ENDPOINT = False

if CONSULTAR_ENDPOINT:
    from databricks.sdk import WorkspaceClient
    workspace = WorkspaceClient()
    resposta = workspace.serving_endpoints.query(
        name=NOME_ENDPOINT,
        dataframe_records=exemplo_entrada.to_dict(orient="records"),
    )
    print(resposta)
else:
    print("Consulta desativada. Ative-a quando o endpoint estiver Ready.")

## 7. Teste HTTP com token (como na aula)

Crie antes um secret scope `desloc-api` com a chave `pat` contendo o token. Este teste reproduz o `requests.post` do professor, sem gravar o token no notebook. Primeiro envia 5 linhas históricas; depois, 1 linha com `head(1)`. Execute somente após o endpoint estar Ready.


In [ ]:
import requests
from databricks.sdk import WorkspaceClient

TOKEN = dbutils.secrets.get(scope="desloc-api", key="pat")
HOST = WorkspaceClient().config.host.rstrip("/")
URL_INVOCACAO = f"{HOST}/serving-endpoints/{NOME_ENDPOINT}/invocations"

def consultar_http(n_linhas):
    dados = X.head(n_linhas).to_dict(orient="split")
    resposta_http = requests.post(
        URL_INVOCACAO,
        headers={"Authorization": f"Bearer {TOKEN}"},
        json={"dataframe_split": dados},
        timeout=60,
    )
    resposta_http.raise_for_status()
    return resposta_http.json()


In [ ]:
resultado_5 = consultar_http(5)
print("Previsões para 5 corridas:", resultado_5.get("predictions", resultado_5))


In [ ]:
resultado_1 = consultar_http(1)
print("Previsão para 1 corrida:", resultado_1.get("predictions", resultado_1))


## Interpretação e próxima etapa

- **Verifique** no MLflow o experimento, a assinatura e a versão registrada.
- **Compare** as previsões locais com as respostas do endpoint para as mesmas entradas.
- **Execute** o `08_avaliacao_retreino` para documentar os critérios de desempenho e o risco de vazamento temporal.
- **Substitua** as features derivadas da entrega por features disponíveis na solicitação antes de qualquer uso real de precificação.